# Parcel-Level Voxel-Pattern ISC — Left-Wing Subjects

**Goal**: Measure inter-subject pattern correlation (ISPC) *within each parcel* of the
Schaefer 400 + Tian S3 atlas (432 parcels), for each of the four political-content
conditions, using left-wing subjects only.

**Analysis logic** (Chen et al., 2017, *Nature Neuroscience*, doi:10.1038/nn.4450):

1. For each subject and condition, load the pre-extracted NPZ `(n_posts, n_brain_voxels)`
2. Average across posts → one spatial pattern `(n_brain_voxels,)` per subject
3. For each parcel: slice the voxels belonging to that parcel → `(n_subjects, n_voxels_in_parcel)`
4. Leave-one-out ISC: for each subject i, compute Pearson r between their parcel pattern
   and the mean of all others — **correlation is across voxels** (spatial similarity)
5. Average r across subjects → one ISC value per parcel × condition (432 × 4 values)

**Key difference from `parcellated_leftwing_ispc.ipynb`**:  
That notebook correlates parcel activation *across posts* (timecourse covariance) — one
scalar per parcel per post.  
This notebook correlates voxel activation *across space within each parcel* (pattern
similarity) — one ISC value per parcel.

**Key difference from `roi_ispc_leftwing.ipynb`**:  
That notebook computes one ISC per ROI (2 values per condition).  
This notebook computes one ISC per parcel (432 values per condition), providing
whole-brain spatial coverage.

**Atlas**: Schaefer 2018 400 Parcels (7 Networks) + Tian S3 subcortical (32 regions)  
**Conditions**: AntiLeft · AntiRight · ProLeft · ProRight  
**Permutation test**: deferred — to be agreed with thesis supervisor

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys

sys.path.insert(0, str(Path.cwd().parent / 'src'))

from yy_fmri_kit.event_isc.parcel_pattern_similarity import (
    run_parcel_isc,
    results_to_dataframe,
)
from yy_fmri_kit.event_isc.contrast import parcels_to_nifti
from statsmodels.stats.multitest import fdrcorrection

## 1. Configuration

In [ ]:
ROOT           = Path('/path/to/project/root')  # CHANGE THIS
BEHAVIORAL_CSV = ROOT / 'behavioral_analyses/data/250226/merged_behavioral_bids.csv'

NPZ_DIR    = ROOT / 'data/derivatives/postbypost/parcel_patterns'
ATLAS_NII  = ROOT / 'data/atlases/Schaefer2018_tf_2mm_400Parcels7Networks_plus_TianS3.dseg.nii.gz'
LABELS_TSV = ROOT / 'data/atlases/Schaefer2018_400Parcels7Networks_plus_TianS3_labels.tsv'

OUTPUT_DIR = ROOT / 'data/derivatives/parcel_ispc/leftwing'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TYPES = ['AntiLeft', 'AntiRight', 'ProLeft', 'ProRight']
FDR_Q     = 0.05
TOP_N     = 20   # parcels to show in bar chart

## 2. Load Left-Wing Subjects

In [ ]:
beh_df   = pd.read_csv(BEHAVIORAL_CSV)
beh_df   = beh_df.drop_duplicates(subset='bids_id', keep='first')
subjects = sorted(beh_df[beh_df['political_group'] == 'left']['bids_id'].tolist())

print(f'Left-wing subjects ({len(subjects)}):')
print(subjects)

## 3. Check Extraction Status

NPZ files are produced by `scripts/extract_parcel_patterns.py`.  
Run the cell below to check how many files exist, or execute the script if needed:

```bash
cd /path/to/your/project/root  # CHANGE THIS
python yy-fMRI-kit/scripts/extract_parcel_patterns.py

```

In [ ]:
npzs = sorted(NPZ_DIR.glob('**/*_desc-parcel_patterns.npz'))
print(f'NPZ files found: {len(npzs)}  (expected {len(subjects) * len(RUN_TYPES)} = '
      f'{len(subjects)} subjects x {len(RUN_TYPES)} conditions)')

if npzs:
    sample = np.load(npzs[0], allow_pickle=True)
    print(f'\nSample: {npzs[0].relative_to(NPZ_DIR)}')
    print(f'  data shape          : {sample["data"].shape}  (n_posts, n_brain_voxels)')
    print(f'  n_parcels in labels : {len(sample["parcel_ids"])}')
else:
    print('\nNo NPZ files found — run extract_parcel_patterns.py first.')

## 4. Run Per-Parcel Pattern ISC — All Conditions

In [ ]:
results = run_parcel_isc(
    npz_dir   = NPZ_DIR,
    subjects  = subjects,
    run_types = RUN_TYPES,
)

## 5. Results Table + FDR Correction

FDR correction (Benjamini-Hochberg) is applied across parcels within each condition.
Note: p-values require a permutation test (deferred).  The table below shows ISC values without significance — update once permutation is run.

In [ ]:
df = results_to_dataframe(results)

# Placeholder p-value columns (fill once permutation test is implemented)
df['p_perm'] = np.nan
df['p_fdr']  = np.nan

# Summary per condition: top 10 by ISC
for cond in RUN_TYPES:
    sub = df[df['condition'] == cond].sort_values('isc_mean', ascending=False)
    print(f'\n── {cond} ──  (mean ISC across parcels: {sub["isc_mean"].mean():.4f})')
    print(sub[['parcel_name','isc_mean']].head(10).to_string(index=False))

## 6. Bar Chart — Top Parcels per Condition

Shows the `TOP_N` parcels with the highest mean ISC for each condition.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10), sharey=False)
axes_flat = axes.flatten()
colors    = ['#E07B54', '#5B8DB8', '#6BAE75', '#A97FC4']

for ax, cond, color in zip(axes_flat, RUN_TYPES, colors):
    sub = (df[df['condition'] == cond]
           .sort_values('isc_mean', ascending=False)
           .head(TOP_N))
    ax.barh(sub['parcel_name'][::-1], sub['isc_mean'][::-1],
            color=color, edgecolor='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.8, linestyle='--')
    ax.set_title(f'{cond}  (top {TOP_N} parcels)', fontweight='bold')
    ax.set_xlabel('Mean ISC (Pearson r)')
    ax.tick_params(axis='y', labelsize=7)
    ax.spines[['top','right']].set_visible(False)

fig.suptitle(
    'Per-Parcel Voxel-Pattern ISC — Left-Wing Subjects',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
fig_path = OUTPUT_DIR / 'parcel_ispc_barplot_top20.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved → {fig_path}')
plt.show()

## 7. ISC Distribution Across Parcels

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
colors    = ['#E07B54', '#5B8DB8', '#6BAE75', '#A97FC4']

for ax, cond, color in zip(axes, RUN_TYPES, colors):
    vals = df[df['condition'] == cond]['isc_mean'].dropna()
    ax.hist(vals, bins=40, color=color, edgecolor='k', linewidth=0.3, alpha=0.85)
    ax.axvline(vals.mean(), color='k', linewidth=1.5, linestyle='--',
               label=f'mean={vals.mean():.3f}')
    ax.set_title(cond, fontweight='bold')
    ax.set_xlabel('ISC (Pearson r)')
    ax.legend(fontsize=8)
    ax.spines[['top','right']].set_visible(False)
axes[0].set_ylabel('Number of parcels')

fig.suptitle('Distribution of per-parcel ISC across 432 parcels', fontsize=12)
plt.tight_layout()
dist_path = OUTPUT_DIR / 'parcel_ispc_distribution.png'
fig.savefig(dist_path, dpi=150, bbox_inches='tight')
print(f'Saved → {dist_path}')
plt.show()

## 8. Build ISC NIfTI Maps

Paint the mean ISC value for each parcel into its voxels using `parcels_to_nifti`
from `contrast.py`.  One NIfTI per condition.

In [ ]:
NIFTI_DIR = OUTPUT_DIR / 'nifti_maps'
NIFTI_DIR.mkdir(exist_ok=True)

nifti_paths = {}

for cond in RUN_TYPES:
    if cond not in results:
        print(f'{cond}: no results — skipping')
        continue

    r            = results[cond]
    parcel_names = [str(n) for n in r['parcel_names']]
    isc_values   = r['isc_mean'].astype(float)   # (n_parcels,)

    out_path = NIFTI_DIR / f'parcel_isc_{cond}.nii.gz'
    parcels_to_nifti(
        values       = isc_values,
        parcel_names = parcel_names,
        atlas_nii    = ATLAS_NII,
        labels_tsv   = LABELS_TSV,
        output_path  = out_path,
    )
    nifti_paths[cond] = out_path
    print(f'{cond}: saved → {out_path.name}')

print(f'\n{len(nifti_paths)} NIfTI maps built')

## 9. Yabplot — Static Surface Brain Maps

Project each condition's ISC NIfTI onto the fsLR-32k midthickness surface.  
Colour scale is shared across all conditions (symmetric ± max |ISC|).

In [ ]:
import yabplot as yab
import yabplot.data as ydata

_lh_surf, _rh_surf = ydata.get_surface_paths('midthickness', 'bmesh')

ALL_VIEWS = [
    'left_lateral', 'left_medial', 'right_lateral', 'right_medial',
    'superior', 'inferior', 'anterior', 'posterior',
]

# Shared symmetric colour scale driven by max |ISC| across all conditions
all_isc = [float(v) for cond in results for v in results[cond]['isc_mean'] if not np.isnan(v)]
VMAX    = max(abs(v) for v in all_isc)
VMAX    = max(VMAX, 0.01)
VMINMAX = [-VMAX, VMAX]
print(f'Shared colour scale: [{-VMAX:.4f}, {VMAX:.4f}]')

STATIC_DIR = OUTPUT_DIR / 'brain_maps_static'
STATIC_DIR.mkdir(exist_ok=True)

for cond, nii_path in nifti_paths.items():
    lh_data, rh_data = yab.project_vol2surf(str(nii_path), interpolation='nearest')
    lh_mesh, rh_mesh = yab.load_vertexwise_mesh(_lh_surf, _rh_surf, lh_data, rh_data)
    yab.plot_vertexwise(
        lh_mesh, rh_mesh,
        views        = ALL_VIEWS,
        cmap         = 'Reds',
        vminmax      = [0.1, 0.7],
        figsize      = (1600, 800),
        display_type = 'static',
        export_path  = str(STATIC_DIR / f'parcel_isc_{cond}.png'),
    )
    print(f'{cond}: saved → parcel_isc_{cond}.png')

## 10. Nilearn — Interactive HTML Brain Maps

Generates an interactive HTML viewer per condition.  Parcels with ISC below `EPS` are transparent (background).

In [ ]:
import nibabel as nib
from nilearn import plotting
from IPython.display import IFrame, display

HTML_DIR = OUTPUT_DIR / 'brain_maps_html'
HTML_DIR.mkdir(exist_ok=True)

EPS = 1e-6

for cond, nii_path in nifti_paths.items():
    img = nib.load(str(nii_path))
    html_view = plotting.view_img(
        img,
        bg_img         = 'MNI152',
        cmap           = 'Reds',
        threshold      = 0.2,
        vmin           = 0.2,
        vmax           = 0.7,
        title          = f'Per-Parcel Pattern ISC — {cond} (left-wing)',
        symmetric_cmap = False,
    )
    out_path = HTML_DIR / f'parcel_isc_{cond}.html'
    html_view.save_as_html(str(out_path))
    print(f'{cond}: saved → {out_path.name}')

# Display first condition inline
display(IFrame(str(HTML_DIR / f'parcel_isc_{RUN_TYPES[0]}.html'), width='100%', height=500))

## 11. Save Results

In [ ]:
# Summary table (one row per condition × parcel)
out_csv = OUTPUT_DIR / 'parcel_ispc_leftwing_summary.csv'
df.to_csv(out_csv, index=False)
print(f'Summary saved → {out_csv}')

# Per-subject ISC (one file per condition)
for cond in RUN_TYPES:
    if cond not in results:
        continue
    r    = results[cond]
    rows = []
    for p_idx, (pid, pname) in enumerate(zip(r['parcel_ids'], r['parcel_names'])):
        for sub, val in zip(r['subjects'], r['isc_subj'][:, p_idx]):
            rows.append({'condition': cond, 'parcel_id': int(pid),
                         'parcel_name': str(pname), 'subject': sub,
                         'isc_r': round(float(val), 6)})
    subj_df  = pd.DataFrame(rows)
    subj_csv = OUTPUT_DIR / f'parcel_ispc_{cond}_persubject.csv'
    subj_df.to_csv(subj_csv, index=False)
    print(f'Per-subject saved → {subj_csv.name}')

df

---

## Approach B: Post-wise ISC (Chen et al. 2017)

In Approach A (sections above), each subject's 18 post-patterns are averaged into
one condition-level mean *before* the inter-subject correlation is computed.

In **Approach B**, a separate leave-one-out ISC is computed for every post
(correlating each subject's voxel pattern for that post against the group mean
for that post), and the resulting per-post ISC values are averaged across posts.

This matches the structure in Chen et al. (2017): pattern vectors are correlated
per scene (post) between subjects, and scene-level correlations are averaged.

| | Approach A | Approach B |
|---|---|---|
| Step 1 | Average 18 posts → one mean pattern | Keep 18 patterns separately |
| Step 2 | LOO Pearson r across voxels | LOO Pearson r per post, then average |
| SNR | Higher (averaging removes noise) | Lower (per-post noise retained) |
| Post-specificity | None (condition-level) | Preserved |


In [ ]:
from yy_fmri_kit.event_isc.parcel_pattern_similarity import (
    run_parcel_isc_postwise,
    load_subject_patterns_postwise,
    permutation_test_timephase_postwise,
    results_to_dataframe,
)

# ── Load from disk if results already exist, otherwise recompute ──────────
# results_b: dict keyed by run_type, each containing:
#   isc_mean (n_parcels,), isc_subj (n_subjects, n_parcels),
#   parcel_ids, parcel_names, voxel_parcel_labels, subjects
_summary_csv = OUTPUT_DIR / 'parcel_isc_B_summary.csv'
_sub_csvs    = [OUTPUT_DIR / f'parcel_isc_B_{c}_persubject.csv' for c in RUN_TYPES]

if _summary_csv.exists() and all(p.exists() for p in _sub_csvs):
    print(f"Approach B results found on disk — loading from {OUTPUT_DIR.name}/")

    # Atlas metadata (parcel_ids, parcel_names, voxel_parcel_labels)
    # are identical across all subjects/conditions — load from any NPZ
    _ref_npz      = next(NPZ_DIR.glob('**/*_desc-parcel_patterns.npz'))
    _d            = np.load(_ref_npz, allow_pickle=True)
    _vox_labels   = _d['voxel_parcel_labels'].astype(np.int32)
    _parcel_ids   = _d['parcel_ids'].astype(np.int32)
    _parcel_names = _d['parcel_names'].astype(object)
    _pname_list   = [str(n) for n in _parcel_names]

    results_b = {}
    for cond, sub_csv in zip(RUN_TYPES, _sub_csvs):
        subj_df  = pd.read_csv(sub_csv, index_col=0)
        # Re-order columns to match NPZ parcel order (safety check)
        subj_df  = subj_df.reindex(columns=_pname_list)
        isc_subj = subj_df.to_numpy(dtype=np.float64)    # (n_subjects, n_parcels)
        isc_mean = isc_subj.mean(axis=0)                  # (n_parcels,)

        results_b[cond] = {
            'isc_mean'           : isc_mean,
            'isc_subj'           : isc_subj,
            'parcel_ids'         : _parcel_ids,
            'parcel_names'       : _parcel_names,
            'voxel_parcel_labels': _vox_labels,
            'n_subjects'         : isc_subj.shape[0],
            'n_posts'            : None,   # not stored in CSV; unused by downstream cells
            'subjects'           : subj_df.index.tolist(),
        }
        print(f"  {cond}: {isc_subj.shape[0]} subjects × {isc_subj.shape[1]} parcels  "
              f"  ISC range [{np.nanmin(isc_mean):.4f}, {np.nanmax(isc_mean):.4f}]")

else:
    print("No cached Approach B results — running run_parcel_isc_postwise() ...")
    results_b = run_parcel_isc_postwise(
        npz_dir    = NPZ_DIR,
        subjects   = subjects,
        run_types  = RUN_TYPES,
        min_voxels = 5,
    )


## B.1 Results Table

In [ ]:
df_b = results_to_dataframe(results_b)
df_b['approach'] = 'B_postwise'

print("Approach B — post-wise ISC summary (mean ISC across parcels per condition)")
summary = df_b.groupby('condition')['isc_mean'].agg(['mean', 'max', 'min']).round(4)
print(summary)
print()
print(f"Top-5 parcels per condition:")
for cond in RUN_TYPES:
    if cond not in results_b:
        continue
    top = df_b[df_b['condition'] == cond].nlargest(5, 'isc_mean')[['parcel_name','isc_mean']]
    print(f"  {cond}:")
    print(top.to_string(index=False))
    print()


## B.2 ISC Distribution Across Parcels

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
colors = ['#E07B54', '#5B8DB8', '#6AA96B', '#9B6BB5']

for ax, (cond, color) in zip(axes, zip(RUN_TYPES, colors)):
    if cond not in results_b:
        ax.set_visible(False)
        continue
    vals = results_b[cond]['isc_mean']
    valid = vals[~np.isnan(vals)]
    ax.hist(valid, bins=40, color=color, alpha=0.75, edgecolor='white')
    ax.axvline(valid.mean(), color='black', lw=1.5, ls='--',
               label=f'mean={valid.mean():.3f}')
    ax.set_title(cond, fontsize=12)
    ax.set_xlabel('ISC (r)', fontsize=10)
    ax.legend(fontsize=9)

axes[0].set_ylabel('Parcel count', fontsize=10)
fig.suptitle('Approach B: Post-wise ISC — Distribution Across Parcels', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'parcel_isc_B_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## B.3 Build ISC NIfTI Maps

In [ ]:
from yy_fmri_kit.event_isc.contrast import parcels_to_nifti

NIFTI_DIR_B = OUTPUT_DIR / 'nifti_maps_B'
NIFTI_DIR_B.mkdir(exist_ok=True)

nifti_paths_b = {}
for cond, r in results_b.items():
    out_path = NIFTI_DIR_B / f'parcel_isc_B_{cond}.nii.gz'
    nii = parcels_to_nifti(
        r['isc_mean'], r['parcel_names'], ATLAS_NII, LABELS_TSV, out_path
    )
    nifti_paths_b[cond] = out_path
    valid = r['isc_mean'][~np.isnan(r['isc_mean'])]
    print(f'  {cond}: saved {out_path.name}  '
          f'(range [{valid.min():.4f}, {valid.max():.4f}])')


In [ ]:
import yabplot as yab
import yabplot.data as ydata

# Reuse _lh_surf, _rh_surf and ALL_VIEWS already defined in the Approach A yabplot cell above

B_VMAX = max(
    float(np.nanmax(np.abs(results_b[c]['isc_mean'])))
    for c in RUN_TYPES if c in results_b
)
B_VMAX    = max(B_VMAX, 0.01)
VMINMAX_B = [-B_VMAX, B_VMAX]
print(f'Approach B colour scale: [{-B_VMAX:.4f}, {B_VMAX:.4f}]')

STATIC_DIR_B = OUTPUT_DIR / 'brain_maps_static_B'
STATIC_DIR_B.mkdir(exist_ok=True)

for cond, nii_path in nifti_paths_b.items():
    lh_data, rh_data = yab.project_vol2surf(str(nii_path), interpolation='nearest')
    lh_mesh, rh_mesh = yab.load_vertexwise_mesh(_lh_surf, _rh_surf, lh_data, rh_data)
    yab.plot_vertexwise(
        lh_mesh, rh_mesh,
        views        = ALL_VIEWS,
        cmap         = 'Reds',
        vminmax      = [0, 0.4],
        figsize      = (1600, 800),
        display_type = 'static',
        export_path  = str(STATIC_DIR_B / f'parcel_isc_B_{cond}.png'),
    )
    print(f'{cond}: saved → parcel_isc_B_{cond}.png')


In [ ]:
from nilearn import plotting
from IPython.display import IFrame, display

HTML_DIR_B = OUTPUT_DIR / 'brain_maps_html_B_reds'
HTML_DIR_B.mkdir(exist_ok=True)
EPS = 1e-6

for cond, nii_path in nifti_paths_b.items():
    img = nib.load(str(nii_path))
    html_view = plotting.view_img(
        img,
        bg_img         = 'MNI152',
        cmap           = 'Reds',
        threshold      = 0.1,
        vmin           = 0.1,
        vmax           = 0.4,
        title          = f'Per-Parcel Pattern ISC (Approach B) — {cond} (left-wing)',
        symmetric_cmap = False,
    )
    out_path = HTML_DIR_B / f'parcel_isc_B_{cond}.html'
    html_view.save_as_html(str(out_path))
    print(f'{cond}: saved → {out_path.name}')

display(IFrame(str(HTML_DIR_B / f'parcel_isc_B_{RUN_TYPES[0]}.html'), width=900, height=500))



---

## Approach A vs B: Comparison

Scatter plot of Approach A (mean-pattern ISC) vs Approach B (post-wise ISC) for
every parcel × condition.  Points on the diagonal indicate perfect agreement.

- A systematic upward shift of A relative to B is expected (averaging before
  correlating inflates r by reducing noise).
- Similar rank ordering across parcels would confirm that both methods identify
  the same brain regions as showing the strongest inter-subject pattern similarity.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = ['#E07B54', '#5B8DB8', '#6AA96B', '#9B6BB5']

for ax, (cond, color) in zip(axes, zip(RUN_TYPES, colors)):
    if cond not in results or cond not in results_b:
        ax.set_visible(False)
        continue
    a_vals = results[cond]['isc_mean']
    b_vals = results_b[cond]['isc_mean']
    valid  = ~(np.isnan(a_vals) | np.isnan(b_vals))
    ax.scatter(a_vals[valid], b_vals[valid],
               alpha=0.35, s=7, color=color, linewidths=0)
    lim = max(np.abs(a_vals[valid]).max(), np.abs(b_vals[valid]).max()) * 1.1
    ax.plot([-lim, lim], [-lim, lim], 'k--', lw=0.8, alpha=0.6, label='identity')
    ax.axhline(0, color='gray', lw=0.5, alpha=0.4)
    ax.axvline(0, color='gray', lw=0.5, alpha=0.4)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel('Approach A (mean-pattern ISC)', fontsize=9)
    ax.set_ylabel('Approach B (post-wise ISC)', fontsize=9)
    ax.set_title(cond, fontsize=11)
    r = np.corrcoef(a_vals[valid], b_vals[valid])[0, 1]
    ax.text(0.05, 0.92, f'r = {r:.3f}', transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

fig.suptitle('Approach A vs B: ISC per Parcel (all conditions)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'parcel_isc_AB_scatter.png', dpi=150, bbox_inches='tight')
plt.show()


## Save Approach B Results

In [ ]:
out_b = OUTPUT_DIR / 'parcel_isc_B_summary.csv'
df_b.to_csv(out_b, index=False)
print(f'Saved {out_b.name}')

for cond, r in results_b.items():
    subj_df = pd.DataFrame(
        r['isc_subj'],
        columns=r['parcel_names'],
        index=r['subjects'],
    )
    out_subj = OUTPUT_DIR / f'parcel_isc_B_{cond}_persubject.csv'
    subj_df.to_csv(out_subj)
    print(f'  Saved {out_subj.name}')


---

## B.4 Permutation Test — Time-Domain Phase Randomization

**Why time-domain?** Phase randomization must be applied to the raw BOLD
timeseries *before* pattern extraction, not to the already-extracted 18-post
patterns. This is the rigorous null because it also controls for temporal
autocorrelation in the BOLD signal (HRF shape, scanner drift, physiological
noise) that could inflate ISC independently of stimulus content.

**Pipeline (per permutation)**:
1. Load each subject's denoised BOLD `(n_trs, n_brain_voxels)` (same
   NiftiMasker settings as the original extraction: `detrend=True`,
   `standardize=False`, atlas brain mask).
2. FFT along the time axis → apply independent random phase shifts per
   subject (one phase per frequency component, broadcast across voxels to
   preserve spatial covariance) → IFFT.
3. Re-extract per-post patterns from the phase-randomised BOLD using the
   same TR windows (onset + 4 TR HRF shift).
4. Compute post-wise LOO ISC per parcel (Pearson r across voxels, averaged
   over posts).

**p-value**: proportion of null ISC ≥ observed (one-tailed, ISC > 0).

> ⚠️ BOLD preloading (~1–4 GB per condition) is done once in the cell below.
> Permutations: ~15–40 min for 1000 iterations × 432 parcels on CPU.

In [ ]:
import nibabel as nib
from nilearn import image
from nilearn.input_data import NiftiMasker

DENOISED_DIR = ROOT / 'data/derivatives/denoised'
EVENTS_CSV   = ROOT / 'behavioral_analyses/data/130426/summary_with_bids_ids.csv'
TR       = 1.0    # seconds — confirmed from NIfTI header
SHIFT_TR = 4      # HRF delay in TRs

# Build the same atlas brain mask used during extraction
atlas_img      = image.load_img(str(ATLAS_NII))
brain_mask_img = image.math_img("img > 0", img=atlas_img)

events_df = pd.read_csv(EVENTS_CSV)

bold_all = {}   # {run_type: {bold_data, onset_trs, offset_trs, vox_labels, parcel_ids, ...}}

for run_type in RUN_TYPES:
    print(f"\n{'='*50}")
    print(f"Loading BOLD for condition: {run_type}")

    # Get common post IDs and subject order from NPZ files (already aligned)
    _, post_ids_arr, vox_labels, parcel_ids, parcel_names, loaded_subs = \
        load_subject_patterns_postwise(NPZ_DIR, subjects, run_type)
    common_posts = post_ids_arr.tolist()
    n_posts      = len(common_posts)

    bold_list   = []
    onset_list  = []
    offset_list = []
    ok_subs     = []

    masker        = NiftiMasker(mask_img=brain_mask_img, detrend=True,
                                standardize=False, resampling_target='data')
    masker_fitted = False

    for sub in loaded_subs:
        func_dir = DENOISED_DIR / sub / 'func'
        niftis   = sorted(func_dir.glob(f'*task-{run_type}*bold*.nii*'))
        if not niftis:
            print(f"  WARNING: no NIfTI for {sub}/{run_type} — skipped")
            continue

        bold_img = image.load_img(str(niftis[0]))
        if not masker_fitted:
            brain_ts     = masker.fit_transform(bold_img).astype(np.float32)
            masker_fitted = True
        else:
            brain_ts = masker.transform(bold_img).astype(np.float32)
        # brain_ts: (n_trs, n_brain_voxels)

        # Compute TR windows for each common post
        sub_ev = events_df[(events_df['bids_id'] == sub) & (events_df['run'] == run_type)]
        n_trs  = brain_ts.shape[0]
        onset_row  = []
        offset_row = []
        for post_id in common_posts:
            row = sub_ev[sub_ev['post_id'] == post_id]
            if len(row) == 0:
                onset_row.append(0);  offset_row.append(1)   # fallback — will be skipped
                print(f"  WARNING: {sub}/{run_type}/{post_id} not in events")
            else:
                onset_s    = float(row['onset_s'].iloc[0])
                duration_s = float(row['duration_s'].iloc[0])
                start = int(round(onset_s / TR)) + SHIFT_TR
                end   = int(round((onset_s + duration_s) / TR)) + SHIFT_TR
                start = max(0, min(start, n_trs - 1))
                end   = max(start + 1, min(end, n_trs))
                onset_row.append(start);  offset_row.append(end)

        bold_list.append(brain_ts)
        onset_list.append(onset_row)
        offset_list.append(offset_row)
        ok_subs.append(sub)

    bold_data = np.stack(bold_list)      # (n_subjects, n_trs, n_brain_voxels)
    onset_trs  = np.array(onset_list)   # (n_subjects, n_posts)
    offset_trs = np.array(offset_list)

    n_s, n_t, n_v = bold_data.shape
    mem_gb = n_s * n_t * n_v * 4 / 1e9
    print(f"  Loaded: {n_s} subjects × {n_t} TRs × {n_v} voxels  "
          f"({mem_gb:.2f} GB float32)")

    bold_all[run_type] = {
        'bold_data'   : bold_data,
        'onset_trs'   : onset_trs,
        'offset_trs'  : offset_trs,
        'vox_labels'  : vox_labels,
        'parcel_ids'  : parcel_ids,
        'parcel_names': parcel_names,
        'common_posts': common_posts,
        'subjects'    : ok_subs,
    }

print("\nAll conditions loaded.")

In [ ]:
import time

N_PERMS = 1000
SEED    = 42

perm_results = {}

for run_type in RUN_TYPES:
    b = bold_all[run_type]
    print(f"\n{'='*50}")
    print(f"Condition: {run_type}  |  {b['bold_data'].shape[0]} subjects  "
          f"|  {b['bold_data'].shape[1]} TRs  |  {b['bold_data'].shape[2]} voxels")
    print(f"Running {N_PERMS} time-domain phase-randomization permutations ...")

    t0 = time.time()
    obs_isc, p_vals, null_dist = permutation_test_timephase_postwise(
        bold_data           = b['bold_data'],
        onset_trs           = b['onset_trs'],
        offset_trs          = b['offset_trs'],
        voxel_parcel_labels = b['vox_labels'],
        parcel_ids          = b['parcel_ids'],
        n_perms             = N_PERMS,
        seed                = SEED,
        min_voxels          = 5,
        verbose             = True,
    )
    elapsed = time.time() - t0
    print(f"Done in {elapsed/60:.1f} min")
    print(f"  ISC range: [{np.nanmin(obs_isc):.4f}, {np.nanmax(obs_isc):.4f}]")
    print(f"  p range:   [{np.nanmin(p_vals):.4f}, {np.nanmax(p_vals):.4f}]")

    perm_results[run_type] = {
        'obs_isc'     : obs_isc,
        'p_vals'      : p_vals,
        'null_dist'   : null_dist,
        'parcel_names': b['parcel_names'],
        'parcel_ids'  : b['parcel_ids'],
        'n_subjects'  : b['bold_data'].shape[0],
        'n_posts'     : b['onset_trs'].shape[1],
    }

## B.5 FDR Correction and Significance Summary

Benjamini–Hochberg FDR correction (q = 0.05) applied separately per condition
across all valid parcels.

In [ ]:
from statsmodels.stats.multitest import fdrcorrection

sig_results = {}

print(f"{'Condition':<12} {'Valid':>6} {'p<0.05 raw':>11} {'FDR sig (q=0.05)':>17}")
print("-" * 50)

for run_type in RUN_TYPES:
    r       = perm_results[run_type]
    obs_isc = r['obs_isc']
    p_vals  = r['p_vals']
    pnames  = np.array([str(n) for n in r['parcel_names']])

    valid_mask = ~np.isnan(p_vals)
    rejected   = np.zeros(len(p_vals), dtype=bool)
    p_fdr      = np.full(len(p_vals), np.nan)

    if valid_mask.sum() > 0:
        rej_v, p_fdr_v = fdrcorrection(p_vals[valid_mask], alpha=FDR_Q)
        rejected[valid_mask] = rej_v
        p_fdr[valid_mask]    = p_fdr_v

    print(f"{run_type:<12} {valid_mask.sum():>6} {int((p_vals < 0.05).sum()):>11} {int(rejected.sum()):>17}")

    sig_results[run_type] = {
        'rejected'        : rejected,
        'p_fdr'           : p_fdr,
        'sig_parcel_names': pnames[rejected].tolist(),
        'sig_isc'         : obs_isc[rejected],
    }

print()
for run_type in RUN_TYPES:
    sr = sig_results[run_type]
    if len(sr['sig_parcel_names']) == 0:
        print(f"{run_type}: no FDR-significant parcels")
        continue
    idx = np.argsort(sr['sig_isc'])[::-1]
    print(f"\n{run_type}  ({len(sr['sig_parcel_names'])} FDR-significant parcels):")
    for name, val in zip(np.array(sr['sig_parcel_names'])[idx], sr['sig_isc'][idx]):
        print(f"  {name:<60}  ISC = {val:.4f}")

## B.6 Brain Maps — FDR-Significant Parcels

Two variants per condition (shared colour scale):
- **All parcels** coloured by observed ISC
- **FDR-significant only** — non-significant parcels set to NaN (light grey)

In [ ]:
import os
import tempfile

try:
    _lh_surf
except NameError:
    import yabplot.data as ydata
    _lh_surf, _rh_surf = ydata.get_surface_paths('midthickness', 'bmesh')

ALL_VIEWS = [
    'left_lateral', 'left_medial', 'right_lateral', 'right_medial',
    'superior', 'inferior', 'anterior', 'posterior',
]

SIG_MAP_DIR = OUTPUT_DIR / 'brain_maps_sig_B'
SIG_MAP_DIR.mkdir(exist_ok=True)

_ref        = perm_results[RUN_TYPES[0]]
_pnames_ref = [str(n) for n in _ref['parcel_names']]

isc_vmax = max(float(np.nanmax(perm_results[c]['obs_isc'])) for c in RUN_TYPES if c in perm_results)
isc_vmax = max(isc_vmax, 0.01)
print(f"Colour scale: [0, {isc_vmax:.4f}]")

def _tmp_nifti(values):
    tmp = tempfile.mktemp(suffix='.nii.gz')
    parcels_to_nifti(values, _pnames_ref, ATLAS_NII, LABELS_TSV, tmp)
    return tmp

def _yab_cortical(nii_path, out_png, vminmax):
    import yabplot as yab
    lh, rh = yab.project_vol2surf(nii_path, interpolation='nearest')
    lm, rm = yab.load_vertexwise_mesh(_lh_surf, _rh_surf, lh, rh)
    yab.plot_vertexwise(lm, rm, views=ALL_VIEWS, cmap='Reds',
                        vminmax=vminmax, figsize=(1600, 800),
                        display_type='static', export_path=str(out_png),
                        nan_color=(0.92, 0.92, 0.92))

for run_type in RUN_TYPES:
    obs_isc  = perm_results[run_type]['obs_isc']
    rejected = sig_results[run_type]['rejected']

    # All-parcels map
    t = _tmp_nifti(obs_isc)
    _yab_cortical(t, SIG_MAP_DIR / f'isc_B_{run_type}_all.png', [0, isc_vmax])
    os.unlink(t)

    # FDR-significant only
    t = _tmp_nifti(np.where(rejected, obs_isc, np.nan))
    _yab_cortical(t, SIG_MAP_DIR / f'isc_B_{run_type}_fdr_sig.png', [0, isc_vmax])
    os.unlink(t)

    print(f"{run_type}: saved  ({int(rejected.sum())} FDR-sig parcels)")

## B.7 Save Results with Significance

In [ ]:
PERM_DIR = OUTPUT_DIR / 'permutation_B'
PERM_DIR.mkdir(exist_ok=True)

all_rows = []

for run_type in RUN_TYPES:
    if run_type not in perm_results or run_type not in sig_results:
        continue

    r        = perm_results[run_type]
    sr       = sig_results[run_type]
    pnames   = [str(n) for n in r['parcel_names']]
    pids     = r['parcel_ids']
    obs_isc  = r['obs_isc']
    p_vals   = r['p_vals']
    p_fdr    = sr['p_fdr']
    rejected = sr['rejected']

    for p_idx, (pid, pname) in enumerate(zip(pids, pnames)):
        all_rows.append({
            'condition'  : run_type,
            'parcel_id'  : int(pid),
            'parcel_name': pname,
            'isc_mean'   : round(float(obs_isc[p_idx]), 6),
            'p_perm'     : round(float(p_vals[p_idx]), 6) if not np.isnan(p_vals[p_idx]) else np.nan,
            'p_fdr'      : round(float(p_fdr[p_idx]),  6) if not np.isnan(p_fdr[p_idx])  else np.nan,
            'significant': bool(rejected[p_idx]),
        })

    null_path = PERM_DIR / f'null_dist_B_{run_type}.npy'
    np.save(str(null_path), r['null_dist'])
    print(f"  Null distribution saved → {null_path.name}  shape {r['null_dist'].shape}")

sig_df = pd.DataFrame(all_rows)
csv_path = OUTPUT_DIR / 'parcel_isc_B_significance.csv'
sig_df.to_csv(csv_path, index=False)
print(f"\nSignificance table saved → {csv_path}")
print(f"  {len(sig_df)} rows  |  {sig_df['significant'].sum()} FDR-significant parcels total")
sig_df[sig_df['significant']].sort_values('isc_mean', ascending=False).head(20)

## B.8 Interactive Brain Map — FDR-Significant Parcels Only

Interactive nilearn HTML viewer per condition showing **only FDR-significant parcels** (q = 0.05).
Non-significant parcels are masked to NaN and rendered transparent.
Colour scale is shared across conditions and anchored to the range of significant ISC values.

In [ ]:
import nibabel as nib
import tempfile, os
from nilearn import plotting
from IPython.display import IFrame, display

HTML_SIG_DIR = OUTPUT_DIR / 'brain_maps_sig_html_B'
HTML_SIG_DIR.mkdir(exist_ok=True)

for run_type in RUN_TYPES:
    obs_isc  = perm_results[run_type]['obs_isc']
    rejected = sig_results[run_type]['rejected']
    pnames   = [str(n) for n in perm_results[run_type]['parcel_names']]
    n_sig    = int(rejected.sum())

    sig_isc = np.where(rejected, obs_isc, np.nan)

    # Colour scale anchored to this condition's significant ISPC values
    sig_vals = obs_isc[rejected]
    if sig_vals.size > 0:
        vmin = max(0.0, float(np.nanmin(sig_vals)))
        vmax = float(np.nanmax(sig_vals))
    else:
        vmin, vmax = 0.0, 0.5
    thresh = max(vmin, 1e-4)
    print(f'{run_type}: {n_sig} FDR-sig parcels  |  ISPC range [{vmin:.4f}, {vmax:.4f}]')

    tmp = tempfile.mktemp(suffix='.nii.gz')
    parcels_to_nifti(sig_isc, pnames, ATLAS_NII, LABELS_TSV, tmp)

    html_view = plotting.view_img(
        tmp,
        bg_img         = 'MNI152',
        cmap           = 'Reds',
        threshold      = 0.1,
        vmin           = 0.1,
        vmax           = 0.25,
        title          = f'ISC Approach B — {run_type}  [{n_sig} FDR-sig parcels, ISPC range {vmin:.3f}–{vmax:.3f}]',
        symmetric_cmap = False,
    )
    out_path = HTML_SIG_DIR / f'parcel_isc_B_sig_{run_type}.html'
    html_view.save_as_html(str(out_path))
    os.unlink(tmp)


## B.9 Null Distributions — Most Significant Parcels

For each condition the **top 3 most significant parcels** (ranked by raw permutation p-value)
are shown with their null ISC distributions (1 000 time-domain phase-randomization permutations).

- **Red solid line**: observed ISC
- **Grey dashed line**: 95th percentile of the null
- **Tick mark ✓**: parcel passes FDR correction (q = 0.05)

In [ ]:
TOP_N = 3  # most significant parcels per condition

COND_COLORS = {
    'anti_left' : '#E07B54',
    'anti_right': '#5B8DB8',
    'pro_left'  : '#6AA96B',
    'pro_right' : '#9B6BB5',
}

fig, axes = plt.subplots(
    len(RUN_TYPES), TOP_N,
    figsize=(TOP_N * 4.5, len(RUN_TYPES) * 3.2),
    squeeze=False,
)
fig.suptitle(
    'Null Distributions — Top Parcels per Condition\n'
    '(time-domain phase randomization, N=1 000)',
    fontsize=13, y=1.01,
)

for row_i, run_type in enumerate(RUN_TYPES):
    r         = perm_results[run_type]
    sr        = sig_results[run_type]
    obs_isc   = r['obs_isc']
    p_vals    = r['p_vals']
    p_fdr     = sr['p_fdr']
    null_dist = r['null_dist']   # (n_perms, n_parcels)
    pnames    = [str(n) for n in r['parcel_names']]
    rejected  = sr['rejected']
    bar_color = COND_COLORS.get(run_type, '#888888')

    # Rank by raw p-value (ascending); push NaN parcels to end
    sort_key = np.where(~np.isnan(p_vals), p_vals, 2.0)
    top_idxs = np.argsort(sort_key)[:TOP_N]

    for col_i, p_idx in enumerate(top_idxs):
        ax        = axes[row_i, col_i]
        null_vals = null_dist[:, p_idx]
        obs       = float(obs_isc[p_idx])
        pv        = float(p_vals[p_idx]) if not np.isnan(p_vals[p_idx]) else np.nan
        pfdr_val  = float(p_fdr[p_idx])  if not np.isnan(p_fdr[p_idx])  else np.nan
        pname     = pnames[p_idx]
        is_sig    = bool(rejected[p_idx])

        # Null histogram
        ax.hist(null_vals, bins=40, color=bar_color, alpha=0.55, edgecolor='none', density=True)

        # 95th percentile of null
        p95 = float(np.percentile(null_vals, 95))
        ax.axvline(p95, color='#888888', linestyle='--', linewidth=1.2, label=f'95th: {p95:.3f}')

        # Observed ISC
        obs_color = '#CC0000' if is_sig else '#CC6600'
        ax.axvline(obs, color=obs_color, linestyle='-', linewidth=2.2,
                   label=f'obs: {obs:.3f}')

        # p-value annotation
        pv_str   = f'{pv:.4f}'   if not np.isnan(pv)      else 'n/a'
        pfdr_str = f'{pfdr_val:.4f}' if not np.isnan(pfdr_val) else 'n/a'
        sig_mark = '  ✓ FDR' if is_sig else ''
        ax.text(
            0.97, 0.97,
            f'p_perm = {pv_str}\np_fdr  = {pfdr_str}{sig_mark}',
            transform=ax.transAxes, ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.80),
        )

        # Title: last 4 underscore-segments split onto two lines
        parts = pname.split('_')
        if len(parts) >= 4:
            short = '_'.join(parts[-4:-2]) + '\n' + '_'.join(parts[-2:])
        elif len(parts) >= 2:
            short = '_'.join(parts[-2:])
        else:
            short = pname
        ax.set_title(short, fontsize=8, pad=3, linespacing=1.3)
        ax.set_xlabel('ISC (null)', fontsize=8)
        if col_i == 0:
            ax.set_ylabel(run_type.replace('_', ' '), fontsize=9, labelpad=4)
        ax.tick_params(labelsize=7)
        ax.legend(fontsize=7, loc='upper left', framealpha=0.75)

plt.tight_layout()
null_fig_path = OUTPUT_DIR / 'null_distributions_B_top_parcels.png'
plt.savefig(str(null_fig_path), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {null_fig_path.name}')